# Lecture 11 examples: Bayesian inference by conditional simulation

STAT-S681 · Simulation-Based Inference


# Three possible coins


Simulate $\theta\in\{1/4,1/2,3/4\}$ and two coin flips.


In [ ]:
import numpy as np
rng = np.random.default_rng(681)

def simulate_three_coin(prior_probs=(1/3, 1/3, 1/3)):
    values = [0.25, 0.5, 0.75]
    theta = rng.choice(values, p=prior_probs)
    X = rng.binomial(1, theta, 2)
    return {"theta": theta, "X": X}


Keep only HH, coded as $X=(1,1)$.


In [ ]:
def rejection_sample_three_coin(prior_probs=(1/3, 1/3, 1/3)):
    while True:
        run = simulate_three_coin(prior_probs)
        if list(run["X"]) == [1, 1]:
            return run["theta"]

rejection_sample_three_coin()


Estimate the posterior probabilities from accepted draws.


In [ ]:
draws = np.array(
    [rejection_sample_three_coin()
     for _ in range(5000)])

# Fraction of accepted draws
# at each value:
for theta in [0.25, 0.5, 0.75]:
    frac = np.mean(draws == theta)
    print(theta, frac)


Repeat with prior probabilities $(0.6,0.3,0.1)$.


In [ ]:
new_prior = [0.6, 0.3, 0.1]
new_draws = np.array([
    rejection_sample_three_coin(new_prior)
    for _ in range(5000)])
for theta in [0.25, 0.5, 0.75]:
    print(theta, np.mean(new_draws == theta))


# Continuous prior for a coin


Simulate $\theta\sim\operatorname{Uniform}(0,1)$ and six flips.


In [ ]:
import numpy as np
rng = np.random.default_rng(681)

def simulate_continuous_coin():
    theta = rng.uniform()
    X = rng.binomial(1, theta, 6)
    return {"theta": theta, "X": X}


Keep the exact observed sequence HHTHHT.


In [ ]:
def rejection_sample_continuous_coin():
    while True:
        run = simulate_continuous_coin()
        if list(run["X"]) == [1, 1, 0, 1, 1, 0]:
            return run["theta"]

rejection_sample_continuous_coin()


Estimate the posterior after matching the complete sequence.


In [ ]:
import matplotlib.pyplot as plt
draws = np.array(
    [rejection_sample_continuous_coin()
     for _ in range(2000)])
print(draws.mean())
plt.hist(draws, bins=40)
plt.xlim(0, 1)
plt.xlabel("theta")
plt.show()


Keep any sequence with four heads instead.


In [ ]:
def rejection_sample_coin_count():
    while True:
        run = simulate_continuous_coin()
        if sum(run["X"]) == 4:
            return run["theta"]

draws_count = np.array(
    [rejection_sample_coin_count()
     for _ in range(2000)])


# Dependent rainy days


Simulate six days; each day after the first persists with probability $1/2$ or redraws using $\theta$.


In [ ]:
import numpy as np
rng = np.random.default_rng(681)

def simulate_rain():
    theta = rng.uniform()
    X = [rng.binomial(1, theta)]
    for day in range(1, 6):
        if rng.uniform() < 0.5:
            X.append(X[-1])                    # weather persists
        else:
            X.append(rng.binomial(1, theta))  # new state
    return {"theta": theta, "X": X}


Use one rejection sampler with a condition $E$ on the six-day sequence.


In [ ]:
def rejection_sample_rain(E):
    while True:
        run = simulate_rain()
        if E(run["X"]):
            return run["theta"]


Match the complete sequences DRRRRD and RDRDRR.


In [ ]:
def E_DRRRRD(X):
    return X == [0, 1, 1, 1, 1, 0]

def E_RDRDRR(X):
    return X == [1, 0, 1, 0, 1, 1]

rejection_sample_rain(E_RDRDRR)


Compare posterior means for the two complete sequences.


In [ ]:
def draws(E):
    return np.array([
        rejection_sample_rain(E)
        for _ in range(2000)])

print(draws(E_DRRRRD).mean())
print(draws(E_RDRDRR).mean())


Condition on four rainy days, regardless of order.


In [ ]:
def E_count(X):
    return sum(X) == 4

count_draws = draws(E_count)
print(count_draws.mean())
